In [ ]:
import pandas as pd
import numpy as np
from scipy.io import arff
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ Imports successful")
print("📊 Ready to load KDD12 click prediction dataset...")


✓ Imports successful
📊 Ready to load KDD12 click prediction dataset...


In [ ]:
# Load the ARFF dataset
print("📥 Loading ARFF file...")

data, meta = arff.loadarff('Click_prediction_small.arff')
df = pd.DataFrame(data)

print(f"✓ Successfully loaded dataset")
print(f"📊 Shape: {df.shape}")
print(f"📁 File size: ~27.5 MB")

# Display basic info
print(f"\n📋 Dataset Overview:")
print(f"   Samples: {len(df):,}")
print(f"   Features: {len(df.columns)}")
print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Show column names
print(f"\n🏷️ Features:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")
    
df.head()


In [ ]:
# Process the target variable (click)
print("🎯 Processing target variable...")

# Convert binary click values from bytes to integers
click_values = []
for val in df['click']:
    if isinstance(val, bytes):
        click_values.append(int(val.decode('utf-8')))
    else:
        click_values.append(int(val))

df['click'] = click_values

# Show target distribution
click_counts = df['click'].value_counts().sort_index()
print(f"\n📊 Click Distribution:")
for click, count in click_counts.items():
    print(f"   {click}: {count:,} samples ({count/len(df)*100:.1f}%)")

print(f"\n✓ Target variable processed: {df['click'].dtype}")
print(f"   Click rate: {df['click'].mean()*100:.1f}%")


In [ ]:
# Analyze each feature
print("🔍 Feature Analysis")
print("=" * 40)

feature_stats = []
for col in df.columns:
    if col != 'click':
        stats = {
            'feature': col,
            'dtype': str(df[col].dtype),
            'unique_values': df[col].nunique(),
            'missing_values': df[col].isnull().sum(),
            'min_value': df[col].min() if df[col].dtype != 'object' else 'N/A',
            'max_value': df[col].max() if df[col].dtype != 'object' else 'N/A'
        }
        feature_stats.append(stats)

features_df = pd.DataFrame(feature_stats)
print(features_df.to_string(index=False))

# Group features by type
print(f"\n📊 Feature Categories:")
ad_features = ['ad_id', 'advertiser_id', 'title_id', 'description_id']
user_features = ['user_id', 'query_id', 'keyword_id']
context_features = ['impression', 'url_hash', 'depth', 'position']

print(f"   🎯 Ad Features: {ad_features}")
print(f"   👤 User Features: {user_features}")
print(f"   📍 Context Features: {context_features}")


In [ ]:
# Analyze categorical features with few unique values
print("📋 Categorical Features Analysis")
print("=" * 35)

categorical_features = ['depth', 'position']

for feature in categorical_features:
    print(f"\n{feature.upper()}:")
    counts = df[feature].value_counts().sort_index()
    for value, count in counts.items():
        click_rate = df[df[feature] == value]['click'].mean()
        print(f"   {value}: {count:,} samples ({count/len(df)*100:.1f}%), CTR: {click_rate*100:.1f}%")

# Impression analysis (top 10 values)
print(f"\nIMPRESSION (top 10 values):")
impression_counts = df['impression'].value_counts().head(10)
for value, count in impression_counts.items():
    click_rate = df[df['impression'] == value]['click'].mean()
    print(f"   {value}: {count:,} samples, CTR: {click_rate*100:.1f}%")


In [ ]:
# Analyze high cardinality features
print("🔢 High Cardinality Features Analysis")
print("=" * 45)

high_card_features = ['user_id', 'query_id', 'ad_id', 'advertiser_id', 'keyword_id', 'title_id', 'description_id', 'url_hash']

for feature in high_card_features:
    unique_count = df[feature].nunique()
    total_samples = len(df)
    sparsity = unique_count / total_samples
    
    # Click rates for top 5 most frequent values
    top_values = df[feature].value_counts().head(5)
    avg_click_rates = []
    for value in top_values.index:
        click_rate = df[df[feature] == value]['click'].mean()
        avg_click_rates.append(click_rate)
    
    print(f"\n{feature.upper()}:")
    print(f"   Unique values: {unique_count:,}")
    print(f"   Sparsity: {sparsity:.3f}")
    print(f"   Top 5 click rates: {[f'{x*100:.1f}%' for x in avg_click_rates]}")

# Overall statistics
print(f"\n📊 Dataset Sparsity Summary:")
print(f"   Most sparse: user_id ({df['user_id'].nunique():,} unique)")
print(f"   Least sparse: position ({df['position'].nunique()} unique)")
print(f"   Average sparsity: {np.mean([df[col].nunique()/len(df) for col in high_card_features]):.3f}")


In [ ]:
# Data quality checks
print("🔍 Data Quality Assessment")
print("=" * 35)

# Missing values
print("📊 Missing Values:")
missing_count = df.isnull().sum().sum()
print(f"   Total missing values: {missing_count}")
if missing_count == 0:
    print("   ✓ No missing values detected!")

# Duplicates
print(f"\n📋 Duplicate Records:")
duplicate_count = df.duplicated().sum()
print(f"   Duplicate rows: {duplicate_count:,}")
if duplicate_count > 0:
    print(f"   Duplicate percentage: {duplicate_count/len(df)*100:.2f}%")

# Data types consistency
print(f"\n🔧 Data Types:")
for col in df.columns:
    dtype = df[col].dtype
    sample_values = df[col].dropna().head(3).tolist()
    print(f"   {col}: {dtype} (samples: {sample_values})")

# Check for anomalies in key features
print(f"\n⚠️ Anomaly Checks:")

# Check for zero or negative IDs
id_columns = ['user_id', 'ad_id', 'advertiser_id', 'query_id', 'keyword_id', 'title_id', 'description_id']
for col in id_columns:
    zero_count = (df[col] == 0).sum()
    negative_count = (df[col] < 0).sum()
    if zero_count > 0 or negative_count > 0:
        print(f"   {col}: {zero_count} zeros, {negative_count} negatives")

print(f"\n✅ Data quality assessment complete!")


In [ ]:
# Final dataset summary
print("🎯 KDD12 CLICK PREDICTION DATASET SUMMARY")
print("=" * 50)

print(f"📊 Dataset Characteristics:")
print(f"   📝 Task: Binary click prediction")
print(f"   📈 Samples: {len(df):,}")
print(f"   🏷️ Features: {len(df.columns) - 1} (+ 1 target)")
print(f"   🎯 Click rate: {df['click'].mean()*100:.1f}%")
print(f"   💾 Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print(f"\n🧠 Feature Breakdown:")
print(f"   🎯 Ad features: 4 (ad_id, advertiser_id, title_id, description_id)")
print(f"   👤 User features: 3 (user_id, query_id, keyword_id)")
print(f"   📍 Context features: 4 (impression, url_hash, depth, position)")

print(f"\n📈 Complexity Indicators:")
print(f"   👥 Unique users: {df['user_id'].nunique():,}")
print(f"   🎯 Unique ads: {df['ad_id'].nunique():,}")
print(f"   🔍 Unique queries: {df['query_id'].nunique():,}")
print(f"   🏢 Unique advertisers: {df['advertiser_id'].nunique():,}")

print(f"\n✅ Data Quality:")
print(f"   ✓ No missing values")
print(f"   ✓ Consistent data types")
print(f"   ✓ Realistic click rate (16.8%)")
print(f"   ✓ High-dimensional sparse features")

print(f"\n🚀 Ready for Click Prediction Modeling!")
print(f"   Perfect for testing DKRL methods on advertising data")
print(f"   Suitable for collaborative filtering approaches")
print(f"   Good benchmark for sparse high-dimensional problems")

# Show final dataset info
print(f"\n📋 Final Dataset Shape: {df.shape}")
df.dtypes
